# Inference Log And Controller Entropy Analysis

Analyze full LVAR model + controller predictions and the separate entropy sidecar. The notebook covers controller behavior, correctness-conditioned entropy, error detection, step trajectories, action-conditioned uncertainty, domains, and diagnostic examples.


In [ ]:
%matplotlib inline

from collections import Counter, defaultdict
import json
from pathlib import Path
from statistics import mean, median

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path("/home/csalt/Haider/DVLM/lvar")]
PREDICTION_RELATIVE_PATHS = [
    Path("outputs/inference/current_lvar_model/m3cot_lvar_predictions.jsonl"),
    Path("outputs/m3cot_lvar_predictions.jsonl"),
]
LOG_PATH = next(
    (root.resolve() / relative for root in ROOT_CANDIDATES for relative in PREDICTION_RELATIVE_PATHS if (root / relative).exists()),
    None,
)
if LOG_PATH is None:
    raise FileNotFoundError("Could not locate m3cot_lvar_predictions.jsonl in the current repo or server repo")
ROOT = next(root.resolve() for root in ROOT_CANDIDATES if LOG_PATH.is_relative_to(root.resolve()))
ENTROPY_PATH = LOG_PATH.with_name(f"{LOG_PATH.stem}_entropy_tracking.json")

ACTION_NAMES = ["THINK", "STOP", "GLOBAL", "REGION", "PATCH"]
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"Predictions: {LOG_PATH}")
print(f"Entropy: {ENTROPY_PATH} (exists={ENTROPY_PATH.exists()})")


## Load JSONL

In [ ]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path):
    rows = []
    bad_rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                bad_rows.append({"line_number": line_number, "error": str(exc)})
    return rows, bad_rows

rows, bad_rows = load_jsonl(LOG_PATH)
entropy_rows = load_json(ENTROPY_PATH) if ENTROPY_PATH.exists() else []
print(f"Rows loaded: {len(rows):,}")
print(f"Entropy rows loaded: {len(entropy_rows):,}")
print(f"Bad rows: {len(bad_rows):,}")
if rows:
    print("Prediction keys:", sorted(rows[0].keys()))
if entropy_rows:
    print("Entropy keys:", sorted(entropy_rows[0].keys()))


## Basic Metrics

In [ ]:
def pct(a, b):
    return a / b if b else 0.0

num_steps = [int(row.get("num_steps") or len(row.get("trace") or [])) for row in rows]
controller_tokens = [int(row.get("num_controller_tokens") or 0) for row in rows]
output_tokens = [int(row.get("num_output_tokens") or 0) for row in rows]
total_tokens = [int(row.get("num_total_tokens") or 0) for row in rows]
correct = [row for row in rows if bool(row.get("correct"))]

print(f"Examples: {len(rows):,}")
print(f"Correct: {len(correct):,}")
print(f"Accuracy: {pct(len(correct), len(rows)):.4f}")
print()
print("Controller steps")
print(f"  min: {min(num_steps) if num_steps else 0}")
print(f"  max: {max(num_steps) if num_steps else 0}")
print(f"  mean: {mean(num_steps) if num_steps else 0:.3f}")
print(f"  median: {median(num_steps) if num_steps else 0:.3f}")
print()
print("Token counts")
print(f"  controller mean: {mean(controller_tokens) if controller_tokens else 0:.3f}")
print(f"  output mean: {mean(output_tokens) if output_tokens else 0:.3f}")
print(f"  total mean: {mean(total_tokens) if total_tokens else 0:.3f}")

## Flatten Trace Steps

In [ ]:
steps = []
for row_index, row in enumerate(rows):
    for step in row.get("trace") or []:
        steps.append({
            "row_index": row_index,
            "example_id": row.get("example_id"),
            "correct": bool(row.get("correct")),
            "domain": row.get("domain"),
            "topic": row.get("topic"),
            "action": str(step.get("action", "UNKNOWN")).upper(),
            "step_idx": step.get("step_idx"),
            "patch_index": step.get("patch_index"),
            "region_index": step.get("region_index"),
            "action_probs": step.get("action_probs") or [],
        })

action_counts = Counter(step["action"] for step in steps)
patch_counts = Counter(step["patch_index"] for step in steps if step["patch_index"] is not None)
region_counts = Counter(step["region_index"] for step in steps if step["region_index"] is not None)

print(f"Trace steps: {len(steps):,}")
print("Action counts:")
for action, count in action_counts.most_common():
    print(f"  {action}: {count:,} ({pct(count, len(steps)):.2%})")
print("\nTop patches:", patch_counts.most_common(10))
print("Top regions:", region_counts.most_common(10))

## Repetition And Transitions

In [ ]:
def pairwise(items):
    return zip(items, items[1:])

repeated_patch_examples = 0
adjacent_repeat_examples = 0
transition_counts = Counter()
stop_examples = 0

for row in rows:
    trace = row.get("trace") or []
    actions = [str(step.get("action", "UNKNOWN")).upper() for step in trace]
    patches = [step.get("patch_index") for step in trace if step.get("patch_index") is not None]
    repeated_patch_examples += int(len(set(patches)) < len(patches))
    adjacent_repeat_examples += int(any(left == right for left, right in pairwise(actions)))
    transition_counts.update(pairwise(actions))
    stop_examples += int("STOP" in actions)

print(f"Examples with repeated patch: {repeated_patch_examples:,} ({pct(repeated_patch_examples, len(rows)):.2%})")
print(f"Examples with adjacent repeated action: {adjacent_repeat_examples:,} ({pct(adjacent_repeat_examples, len(rows)):.2%})")
print(f"Examples with STOP: {stop_examples:,} ({pct(stop_examples, len(rows)):.2%})")
print("\nTop transitions:")
for (left, right), count in transition_counts.most_common(15):
    print(f"  {left} -> {right}: {count:,}")

## Accuracy By Group

In [ ]:
def group_accuracy(key):
    grouped = defaultdict(lambda: {"total": 0, "correct": 0, "steps": []})
    for row in rows:
        group = row.get(key) or "UNKNOWN"
        grouped[group]["total"] += 1
        grouped[group]["correct"] += int(bool(row.get("correct")))
        grouped[group]["steps"].append(int(row.get("num_steps") or len(row.get("trace") or [])))
    return sorted(grouped.items(), key=lambda item: item[1]["total"], reverse=True)

for label, values in group_accuracy("domain")[:20]:
    print(f"{label}: acc={pct(values['correct'], values['total']):.4f} total={values['total']:,} avg_steps={mean(values['steps']):.2f}")

## Plots

In [ ]:
max_steps = max(num_steps) if num_steps else 0
plt.figure()
plt.hist(num_steps, bins=range(0, max_steps + 2), edgecolor="black")
plt.title("Controller steps per example")
plt.xlabel("Number of steps")
plt.ylabel("Examples")
plt.tight_layout()
plt.show()

correct_steps = [int(row.get("num_steps") or len(row.get("trace") or [])) for row in rows if bool(row.get("correct"))]
incorrect_steps = [int(row.get("num_steps") or len(row.get("trace") or [])) for row in rows if not bool(row.get("correct"))]
plt.figure()
if correct_steps:
    plt.hist(correct_steps, alpha=0.6, label="correct", bins=range(0, max_steps + 2), edgecolor="black")
if incorrect_steps:
    plt.hist(incorrect_steps, alpha=0.6, label="incorrect", bins=range(0, max_steps + 2), edgecolor="black")
plt.title("Controller steps by correctness")
plt.xlabel("Number of steps")
plt.ylabel("Examples")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def bar(counter, title, xlabel, top_n=20):
    if not counter:
        print(f"No data for {title}")
        return
    items = Counter(counter).most_common(top_n)
    labels = [str(key) for key, _ in items]
    values = [value for _, value in items]
    plt.figure(figsize=(10, 5))
    plt.bar(labels, values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

bar(action_counts, "Controller action frequency", "Action")
bar(patch_counts, "Top patch indices", "Patch index")
bar(region_counts, "Top region indices", "Region index")

In [ ]:
domain_stats = group_accuracy("domain")[:20]
if domain_stats:
    labels = [str(key) for key, _ in domain_stats]
    values = [pct(v["correct"], v["total"]) for _, v in domain_stats]
    plt.figure(figsize=(11, 5))
    plt.bar(labels, values)
    plt.title("Accuracy by domain")
    plt.xlabel("Domain")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## Controller Entropy Analysis

The entropy sidecar records action-, region-, and patch-head entropy at every controller step. Joint controller entropy is `H(action) + P(REGION) H(region) + P(PATCH) H(patch)`, the entropy of the hierarchical controller choice. All values use natural logarithms and therefore are measured in nats.


In [ ]:
prediction_by_id = {row.get("example_id"): row for row in rows}
entropy_records = []
for entropy_row in entropy_rows:
    example_id = entropy_row.get("example_id")
    prediction = prediction_by_id.get(example_id, {})
    entropy_records.append({
        **entropy_row,
        "correct": bool(entropy_row.get("correct")),
        "domain": prediction.get("domain"),
        "topic": prediction.get("topic"),
        "question": prediction.get("question"),
        "generated_text": prediction.get("generated_text"),
        "num_steps": prediction.get("num_steps") or len(prediction.get("trace") or []),
        "num_controller_tokens": prediction.get("num_controller_tokens"),
        "controller_actions": [str(step.get("action", "UNKNOWN")).upper() for step in (prediction.get("trace") or [])],
    })
controller_df = pd.DataFrame(entropy_records)
controller_metrics = [
    "controller_action_entropy_mean",
    "controller_region_entropy_mean",
    "controller_patch_entropy_mean",
    "controller_entropy_mean",
]
if controller_df.empty:
    print("No entropy sidecar rows were found. Rerun full LVAR inference with controller entropy tracking enabled.")
else:
    for metric in controller_metrics:
        controller_df[metric] = pd.to_numeric(controller_df.get(metric), errors="coerce")
HAS_CONTROLLER_ENTROPY = (
    not controller_df.empty
    and "controller_entropy_mean" in controller_df
    and controller_df["controller_entropy_mean"].notna().any()
)
if HAS_CONTROLLER_ENTROPY:
    print(f"Samples: {len(controller_df):,}")
    print(f"Accuracy: {controller_df['correct'].mean():.2%}")
    display(controller_df[controller_metrics].notna().mean().rename("coverage").to_frame().style.format("{:.2%}"))
elif not controller_df.empty:
    print("The sidecar predates controller entropy tracking. Rerun inference to populate the controller fields.")
controller_df.head()


### Correct vs Incorrect

This compares complete distributions rather than only averages. Differences in action entropy indicate uncertainty about what kind of reasoning operation to perform; region and patch entropy isolate uncertainty about where to look; joint entropy combines those decisions according to the policy hierarchy.


In [ ]:
controller_metric_labels = {
    "controller_action_entropy_mean": "Action",
    "controller_region_entropy_mean": "Region",
    "controller_patch_entropy_mean": "Patch",
    "controller_entropy_mean": "Joint controller",
}
if HAS_CONTROLLER_ENTROPY:
    correctness_entropy_summary = (
        controller_df.groupby("correct")[controller_metrics]
        .agg(["count", "mean", "median", "std", lambda s: s.quantile(0.10), lambda s: s.quantile(0.90)])
        .rename(columns={"<lambda_0>": "p10", "<lambda_1>": "p90"})
    )
    display(correctness_entropy_summary.style.format("{:.4f}"))

    entropy_long = controller_df.melt(
        id_vars=["example_id", "correct"],
        value_vars=controller_metrics,
        var_name="metric",
        value_name="entropy",
    )
    entropy_long["head"] = entropy_long["metric"].map(controller_metric_labels)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    if HAS_SEABORN:
        sns.boxplot(data=entropy_long, x="head", y="entropy", hue="correct", showfliers=False, ax=axes[0])
        sns.histplot(data=controller_df, x="controller_entropy_mean", hue="correct", bins=40, kde=True, stat="density", common_norm=False, element="step", ax=axes[1])
    else:
        entropy_long.pivot_table(index="example_id", columns=["head", "correct"], values="entropy").boxplot(ax=axes[0], rot=30)
        for outcome, group in controller_df.groupby("correct"):
            axes[1].hist(group["controller_entropy_mean"], bins=40, alpha=0.45, density=True, label=f"correct={outcome}")
        axes[1].legend()
    axes[0].set_title("Mean entropy by controller head")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Entropy (nats)")
    axes[1].set_title("Joint controller entropy by correctness")
    axes[1].set_xlabel("Mean entropy across controller steps")
    axes[1].set_ylabel("Density")
    plt.tight_layout()
    plt.show()


### Error Detection and Accuracy by Entropy Quantile

AUC measures whether entropy ranks errors above correct answers: `0.5` is chance and values above `0.5` mean errors tend to have higher entropy. Quintile accuracy reveals whether that relationship is monotonic and therefore potentially useful for abstention or sample triage.


In [ ]:
def error_detection_auc(frame, metric):
    usable = frame[[metric, "correct"]].dropna()
    if usable["correct"].nunique() < 2:
        return np.nan
    error = (~usable["correct"]).astype(int)
    ranks = usable[metric].rank(method="average")
    n_error = int(error.sum())
    n_correct = int((1 - error).sum())
    return float((ranks[error == 1].sum() - n_error * (n_error + 1) / 2) / (n_error * n_correct))

if HAS_CONTROLLER_ENTROPY:
    detection_rows = []
    quantile_rows = []
    for metric, label in controller_metric_labels.items():
        means = controller_df.groupby("correct")[metric].mean()
        detection_rows.append({
            "head": label,
            "incorrect_minus_correct_mean": means.get(False, np.nan) - means.get(True, np.nan),
            "error_detection_auc": error_detection_auc(controller_df, metric),
            "coverage": controller_df[metric].notna().mean(),
        })
        usable = controller_df.dropna(subset=[metric]).copy()
        if usable[metric].nunique() >= 2:
            usable["entropy_quantile"] = pd.qcut(usable[metric], q=min(5, usable[metric].nunique()), duplicates="drop")
            grouped = usable.groupby("entropy_quantile", observed=True).agg(
                samples=("example_id", "size"), accuracy=("correct", "mean"), mean_entropy=(metric, "mean")
            ).reset_index()
            grouped["head"] = label
            grouped["quantile_rank"] = range(1, len(grouped) + 1)
            quantile_rows.append(grouped)
    detection_df = pd.DataFrame(detection_rows)
    display(detection_df.style.format({"incorrect_minus_correct_mean": "{:.4f}", "error_detection_auc": "{:.4f}", "coverage": "{:.2%}"}))

    entropy_quantiles = pd.concat(quantile_rows, ignore_index=True) if quantile_rows else pd.DataFrame()
    if not entropy_quantiles.empty:
        if HAS_SEABORN:
            sns.lineplot(data=entropy_quantiles, x="quantile_rank", y="accuracy", hue="head", marker="o")
        else:
            for head, group in entropy_quantiles.groupby("head"):
                plt.plot(group["quantile_rank"], group["accuracy"], marker="o", label=head)
            plt.legend()
        plt.gca().yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
        plt.xlabel("Entropy quintile (low to high)")
        plt.ylabel("Accuracy")
        plt.title("Accuracy as controller uncertainty increases")
        plt.show()


### Step Trajectories and Selected Actions

Step-level analysis distinguishes early planning uncertainty from uncertainty that emerges after several reasoning operations. The action-conditioned table also reports the relevant index entropy only when REGION or PATCH is selected.


In [ ]:
controller_step_rows = []
for _, sample in (controller_df.iterrows() if HAS_CONTROLLER_ENTROPY else []):
    actions = sample.get("controller_actions") or []
    series = {
        "action_entropy": sample.get("controller_action_entropies") or [],
        "region_entropy": sample.get("controller_region_entropies") or [],
        "patch_entropy": sample.get("controller_patch_entropies") or [],
        "joint_entropy": sample.get("controller_entropies") or [],
    }
    num_trace_steps = max([len(v) for v in series.values()] + [len(actions)])
    for step_idx in range(num_trace_steps):
        action = actions[step_idx] if step_idx < len(actions) else "UNKNOWN"
        record = {"example_id": sample["example_id"], "correct": sample["correct"], "step_idx": step_idx, "action": action}
        for metric, values in series.items():
            record[metric] = values[step_idx] if step_idx < len(values) else np.nan
        record["selected_index_entropy"] = record["region_entropy"] if action == "REGION" else record["patch_entropy"] if action == "PATCH" else np.nan
        controller_step_rows.append(record)
controller_steps_df = pd.DataFrame(controller_step_rows)

if not controller_steps_df.empty:
    trajectory = controller_steps_df.groupby(["step_idx", "correct"])["joint_entropy"].agg(["count", "mean", "std"]).reset_index()
    trajectory["sem"] = trajectory["std"] / np.sqrt(trajectory["count"].clip(lower=1))
    for outcome, group in trajectory.groupby("correct"):
        group = group.sort_values("step_idx")
        plt.plot(group["step_idx"], group["mean"], marker="o", label=f"correct={outcome}")
        plt.fill_between(group["step_idx"], group["mean"] - group["sem"], group["mean"] + group["sem"], alpha=0.18)
    plt.xlabel("Controller step")
    plt.ylabel("Mean joint entropy (nats)")
    plt.title("Controller uncertainty through the trace")
    plt.legend()
    plt.show()

    action_summary = controller_steps_df.groupby(["action", "correct"]).agg(
        steps=("example_id", "size"),
        action_entropy=("action_entropy", "mean"),
        region_entropy=("region_entropy", "mean"),
        patch_entropy=("patch_entropy", "mean"),
        selected_index_entropy=("selected_index_entropy", "mean"),
        joint_entropy=("joint_entropy", "mean"),
    ).reset_index()
    display(action_summary.style.format({c: "{:.4f}" for c in action_summary.columns if c.endswith("entropy")}))


### Trace Length, Domains, and Diagnostic Examples

The final section tests whether entropy is mostly a trace-length effect, identifies domains with unusually uncertain controller behavior, and surfaces low-entropy errors and high-entropy successes for qualitative review.


In [ ]:
if HAS_CONTROLLER_ENTROPY:
    relationship_cols = controller_metrics + ["num_steps", "num_controller_tokens", "num_output_tokens"]
    relationship_cols = [col for col in relationship_cols if col in controller_df]
    display(controller_df[relationship_cols].corr(method="spearman").style.format("{:.3f}").set_caption("Spearman correlations"))

    if HAS_SEABORN:
        sns.scatterplot(data=controller_df, x="num_steps", y="controller_entropy_mean", hue="correct", alpha=0.6)
    else:
        for outcome, group in controller_df.groupby("correct"):
            plt.scatter(group["num_steps"], group["controller_entropy_mean"], alpha=0.6, label=f"correct={outcome}")
        plt.legend()
    plt.xlabel("Controller steps")
    plt.ylabel("Mean joint controller entropy")
    plt.title("Controller entropy vs trace length")
    plt.show()

    domain_entropy = controller_df.groupby("domain", dropna=False).agg(
        samples=("example_id", "size"),
        accuracy=("correct", "mean"),
        mean_joint_entropy=("controller_entropy_mean", "mean"),
        mean_action_entropy=("controller_action_entropy_mean", "mean"),
        mean_region_entropy=("controller_region_entropy_mean", "mean"),
        mean_patch_entropy=("controller_patch_entropy_mean", "mean"),
        mean_steps=("num_steps", "mean"),
    ).reset_index().sort_values("samples", ascending=False)
    display(domain_entropy.style.format({
        "accuracy": "{:.2%}", "mean_joint_entropy": "{:.4f}", "mean_action_entropy": "{:.4f}",
        "mean_region_entropy": "{:.4f}", "mean_patch_entropy": "{:.4f}", "mean_steps": "{:.2f}",
    }))

    diagnostic_cols = ["example_id", "domain", "topic", "gold_answer", "generated_text", "controller_entropy_mean", "controller_action_entropy_mean", "num_steps"]
    print("Confident errors: lowest-entropy incorrect samples")
    display(controller_df[~controller_df["correct"]].nsmallest(15, "controller_entropy_mean")[diagnostic_cols])
    print("Uncertain successes: highest-entropy correct samples")
    display(controller_df[controller_df["correct"]].nlargest(15, "controller_entropy_mean")[diagnostic_cols])
